In [9]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()


Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [11]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [ ]:
from dotenv import load_dotenv
import openai
import os
import numpy as np
import json
from xai_sdk import Client as XAIClient

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_AZURE_OPENAI": "Azure OpenAI",
    "ENDPOINT_AZURE_OPENAI": "Azure OpenAI Endpoint",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok"
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        #raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")
        continue


backend = "openai"   # "openai", "azure_openai", "deepseek", "grok"

if backend == "deepseek":
    client = openai.OpenAI(
        api_key=os.getenv("API_KEY_DEEPSEEK"), 
        base_url="https://api.deepseek.com"
    )
    model = "deepseek-chat"
    model_filename = "deepseek_v3"

elif backend == "holistic":
    client = openai.OpenAI(api_key=os.getenv("API_KEY_AZURE_OPENAI"))
    model = "gpt-4.1-mini"
    model_filename = "openai_4.1_mini"

elif backend == "openai":
    client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
    model = "gpt-4.1-mini"
    model_filename = "openai_4.1_mini"

elif backend == "openai2":
    client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
    model = "gpt-4o-mini"
    model_filename = "openai_4o_mini"

elif backend == "grok":
    client = XAIClient(api_key=os.getenv("XAI_API_KEY"), timeout=3600)
    model = "grok-4"
    model_filename = "grok_4"

else:
    raise ValueError(f"Unknown backend: {backend}")

## Basic Zero-Shot prompting

In [15]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from zero_shot_basic import ZeroShotBasic
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields

case_name_set = ["stereotype"]
prompt_type = "short"
    
if True:
    for case_name in case_name_set:
        if case_name.lower() == "manipulation":
            case = manipulation_case
            data = sample_mentalmanip
            output_file = f"results/{model_filename}/zero_shot_basic/classic/results_manipulation_zero_shot_prompt_short_binary.csv"
 
        elif case_name.lower() == "stereotype":
            case = stereotypes_case
            data = sample_mgsd
            output_file = f"results/{model_filename}/zero_shot_basic/classic/results_stereotype_zero_shot_prompt_short_binary_AA.csv"
 
        else:
            raise ValueError(f"Unknown case name: {case_name}")
 
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
 
        if os.path.exists(output_file):
            df_out_existing = pd.read_csv(output_file)
            done_ids = set(df_out_existing["sample_id"])
            rows = df_out_existing.to_dict(orient="records")
            print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
        else:
            done_ids = set()
            rows = []
 
        zero_shot_classifier = ZeroShotBasic(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
        )
 
        try:
            for idx, row in tqdm(data.iterrows(), total=len(data)):
                if idx in done_ids:
                    continue
 
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
 
                try:
                    predicted_label, stats = zero_shot_classifier.classify(text)
                    mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
                except RateLimitError as e:
                    print(f"\nRate limit hit at sample {idx}. Saving progress.")
                    break
                except Exception as e:
                    print(f"\nError at sample {idx}: {e}. Skipping.")
                    continue
 
                additional = get_additional_fields(row, case_name) 
 
                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "max_tokens": zero_shot_classifier.max_tokens,
                    "tokens_used": stats["tokens_used"],
                    "prompt_tokens": stats["prompt_tokens"],
                    "completion_tokens": stats["completion_tokens"],
                    "latency": stats["latency"],
                    **additional,
                }
 
                rows.append(results)
 
        except KeyboardInterrupt:
            print("=== Interrupted manually. Saving progress...")
 
        finally:
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"✅ Saved {len(df_out)} rows to {output_file}")
 
            if case_name == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name == "stereotype":
                y_true = df_out["true_label"]
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
 
            print("=== Classification Report ===\n")
            print(classification_report(y_true, y_pred))
 
            print("\n=== Confusion Matrix ===\n")
            labels = sorted(set(y_true) | set(y_pred))
            conf_matrix = confusion_matrix(y_true, y_pred)
            print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
 
            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy: {accuracy:.2%} ===")

100%|██████████| 500/500 [04:10<00:00,  2.00it/s]


✅ Saved 500 rows to results/openai_4.1_mini/zero_shot_basic/classic/results_stereotype_zero_shot_prompt_short_binary_AA.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       0.77      0.70      0.73       250
   unrelated       0.73      0.79      0.76       250

    accuracy                           0.75       500
   macro avg       0.75      0.75      0.75       500
weighted avg       0.75      0.75      0.75       500


=== Confusion Matrix ===

            stereotype  unrelated
stereotype         175         75
unrelated           52        198

=== Accuracy: 74.60% ===


## Zero-shot prompting

In [14]:
import pandas as pd
from tqdm import tqdm
import os, json
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from cases.get_case_config import get_case_config
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields

case_name_set = ["stereotype"]
prompt_type = "short"
    
if True:
    for case_name in case_name_set:
        if case_name.lower() == "manipulation":
            case = manipulation_case
            task_definition = manipulation_definition_short
            data = sample_mentalmanip
            output_file = f"results/{model_filename}/zero_shot/classic/results_manipulation_zero_shot_prompt_short_binary.csv"
 
        elif case_name.lower() == "stereotype":
            case = stereotypes_case
            task_definition = stereotype_definition_short_binary
            data = sample_mgsd
            output_file = f"results/{model_filename}/zero_shot/classic/results_stereotype_zero_shot_prompt_short_binary.csv"
 
        else:
            raise ValueError(f"Unknown case name: {case_name}")
 
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
 
        if os.path.exists(output_file):
            df_out_existing = pd.read_csv(output_file)
            done_ids = set(df_out_existing["sample_id"])
            rows = df_out_existing.to_dict(orient="records")
            print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
        else:
            done_ids = set()
            rows = []
 
        zero_shot_classifier = ZeroShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            task_definition=task_definition,
        )
 
        try:
            for idx, row in tqdm(data.iterrows(), total=len(data)):
                if idx in done_ids:
                    continue
 
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
 
                try:
                    predicted_label, stats = zero_shot_classifier.classify(text)
                    mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
                except RateLimitError as e:
                    print(f"\nRate limit hit at sample {idx}. Saving progress.")
                    break
                except Exception as e:
                    print(f"\nError at sample {idx}: {e}. Skipping.")
                    continue
 
                additional = get_additional_fields(row, case_name) 
 
                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "max_tokens": zero_shot_classifier.max_tokens,
                    "tokens_used": stats["tokens_used"],
                    "prompt_tokens": stats["prompt_tokens"],
                    "completion_tokens": stats["completion_tokens"],
                    "latency": stats["latency"],
                    **additional,
                }
 
                rows.append(results)
 
        except KeyboardInterrupt:
            print("=== Interrupted manually. Saving progress...")
 
        finally:
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"✅ Saved {len(df_out)} rows to {output_file}")
 
            if case_name == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name == "stereotype":
                y_true = df_out["true_label"]
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
 
            print("=== Classification Report ===\n")
            print(classification_report(y_true, y_pred))
 
            print("\n=== Confusion Matrix ===\n")
            labels = sorted(set(y_true) | set(y_pred))
            conf_matrix = confusion_matrix(y_true, y_pred)
            print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
 
            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy: {accuracy:.2%} ===")

  1%|          | 6/500 [00:03<04:40,  1.76it/s]

=== Interrupted manually. Saving progress...
✅ Saved 6 rows to results/openai_4.1_mini/zero_shot_new/classic/results_stereotype_zero_shot_prompt_short_binary.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       1.00      1.00      1.00         3
   unrelated       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


=== Confusion Matrix ===

            stereotype  unrelated
stereotype           3          0
unrelated            0          3

=== Accuracy: 100.00% ===


## Few-shots prompting

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from stereotype_definitions import stereotype_definition_short, stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition, manipulation_definition_short
from cases.get_case_config import get_case_config
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from data_loader import get_additional_fields


case_name_set = ["stereotype"] #"manipulation"
prompt_type = "short"


for case_name in case_name_set:
    for j in range(2, 6):
        try: 
            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
                examples_df = sample_examples_mentalmanip
                output_file = f"results/{model_filename}/few_shot/classic/results_manipulation_few_shot_prompt_short_{j}examples.csv"
            
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
                examples_df = sample_examples_mgsd
                output_file = f"results/{model_filename}/few_shot/classic/results_stereotype_few_shot_prompt_short_{j}examples_binary.csv"
            
            else:
                raise ValueError(f"Unknown case name: {case_name}")
            
            
            few_shot_classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                task_definition=task_definition, 
                n_shots=j,
                examples_df=examples_df,
            )
            
            
            rows = []
            
            
            for idx, row in tqdm(data.iterrows(), total=len(data)):
                text = row[case.input_col]
                true_label = row[case.label_col]
                
                if isinstance(true_label, str):
                    true_label = true_label.strip()
            
                predicted_label, stats = few_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            
                additional = get_additional_fields(row, case_name)

                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "max_tokens": few_shot_classifier.max_tokens,
                    "tokens_used": stats["tokens_used"],
                    "prompt_tokens": stats["prompt_tokens"],
                    "completion_tokens": stats["completion_tokens"],
                    "latency": stats["latency"],
                    **additional,
                }
            
                rows.append(results)
            
            
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file}")
            
            if case_name == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            
            elif case_name == "stereotype":
                y_true = df_out["true_label"]
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
            
            
            print("=== Classification Report ===\n")
            print(classification_report(y_true, y_pred))
            
            
            print("\n=== Confusion Matrix ===\n")
            labels = sorted(set(y_true) | set(y_pred))
            conf_matrix = confusion_matrix(y_true, y_pred)
            print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
            
            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy: {accuracy:.2%} ===")
            
        except Exception as e:
            print(f"Error with n_shots={j}: {e}")

100%|██████████| 500/500 [31:23<00:00,  3.77s/it]


=== Saved 500 rows to results/deepseek_v3/few_shot/classic/results_stereotype_few_shot_prompt_short_2examples_binary.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       0.79      0.73      0.76       250
   unrelated       0.75      0.81      0.78       250

    accuracy                           0.77       500
   macro avg       0.77      0.77      0.77       500
weighted avg       0.77      0.77      0.77       500


=== Confusion Matrix ===

            stereotype  unrelated
stereotype         183         67
unrelated           48        202

=== Accuracy: 77.00% ===


100%|██████████| 500/500 [29:54<00:00,  3.59s/it]


=== Saved 500 rows to results/deepseek_v3/few_shot/classic/results_stereotype_few_shot_prompt_short_3examples_binary.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       0.82      0.72      0.76       250
   unrelated       0.75      0.84      0.79       250

    accuracy                           0.78       500
   macro avg       0.78      0.78      0.78       500
weighted avg       0.78      0.78      0.78       500


=== Confusion Matrix ===

            stereotype  unrelated
stereotype         179         71
unrelated           40        210

=== Accuracy: 77.80% ===


100%|██████████| 500/500 [40:51<00:00,  4.90s/it]   


=== Saved 500 rows to results/deepseek_v3/few_shot/classic/results_stereotype_few_shot_prompt_short_4examples_binary.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       0.80      0.72      0.76       250
   unrelated       0.75      0.82      0.78       250

    accuracy                           0.77       500
   macro avg       0.77      0.77      0.77       500
weighted avg       0.77      0.77      0.77       500


=== Confusion Matrix ===

            stereotype  unrelated
stereotype         181         69
unrelated           45        205

=== Accuracy: 77.20% ===


100%|██████████| 500/500 [29:41<00:00,  3.56s/it]

=== Saved 500 rows to results/deepseek_v3/few_shot/classic/results_stereotype_few_shot_prompt_short_5examples_binary.csv
=== Classification Report ===

              precision    recall  f1-score   support

  stereotype       0.81      0.66      0.73       250
   unrelated       0.71      0.85      0.78       250

    accuracy                           0.75       500
   macro avg       0.76      0.75      0.75       500
weighted avg       0.76      0.75      0.75       500


=== Confusion Matrix ===

            stereotype  unrelated
stereotype         165         85
unrelated           38        212

=== Accuracy: 75.40% ===


## Zero-shot and Few-shot (3 examples only) prompting with profile keys

In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS, PERSON_SYSTEMATIC, PERSON_SEEDS_CORE

d = [f"profile{i}" for i in range(26, 30)]
selected_profiles = d
prompt_type = "short"
max_tokens = 300
case_name_set = ["manipulation"]   #"stereotype", 

person_set = PERSON_ETHNICS


person_seeds = person_set.seeds

if person_seeds == PERSON_SEEDS_CORE:
    folder_role_playing = "role_playing_core" 
elif (person_seeds == PERSON_SYSTEMATIC.seeds):
    folder_role_playing = "role_playing_system"
elif (person_seeds == PERSON_ETHNICS.seeds):
    folder_role_playing = "role_playing_ethnics"
else:
    folder_role_playing = "role_playing"


for case_name in case_name_set:

    if case_name == "manipulation":
        case = manipulation_case
        task_definition = manipulation_definition_short
        data = sample_mentalmanip
        few_shot_examples = sample_examples_mentalmanip
    elif case_name == "stereotype":
        case = stereotypes_case
        task_definition = stereotype_definition_short_binary
        data = sample_mgsd
        few_shot_examples = sample_examples_mgsd
    else:
        raise ValueError(f"Unknown case name: {case_name}")

    for person_key in selected_profiles:
        for role_playing in ["passive"]:
            for setting in ["few_shot"]:


                file_suffix = (
                    f"results_{case_name}_{setting}_prompt_short_binary.csv"
                    if setting == "zero_shot"
                    else f"results_{case_name}_{setting}_prompt_short_3examples_binary.csv"
                )
                output_file = f"results/{model_filename}/{setting}/{folder_role_playing}/{person_key}_{role_playing}/{file_suffix}"
                os.makedirs(os.path.dirname(output_file), exist_ok=True)

                if setting == "zero_shot":
                    classifier = ZeroShot(
                        case=case,
                        client=client,
                        model=model,
                        max_tokens=max_tokens,
                        task_definition=task_definition,
                        person_key=person_key,
                        role_playing=role_playing,
                        person_set=person_set
                    )
                else:
                    classifier = FewShot(
                        case=case,
                        client=client,
                        model=model,
                        max_tokens=max_tokens,
                        task_definition=task_definition,
                        n_shots=3,
                        examples_df=few_shot_examples,
                        person_key=person_key,
                        role_playing=role_playing,
                        person_set=person_set
                    )

                rows = []
                for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"{case_name} | {person_key} | {setting} | {role_playing}"):
                    try:
                        text = row[case.input_col]
                        true_label = row[case.label_col]
                        pred, stats = classifier.classify(text)
                        mapped = case.label_map.get(pred.strip(), list(case.label_map.values())[-1])

                        additional = get_additional_fields(row, case_name)

                        rows.append({
                            "sample_id": idx,
                            "text": text,
                            "true_label": true_label,
                            "pred_label": mapped,
                            "max_tokens": classifier.max_tokens,
                            "tokens_used": stats["tokens_used"],
                            "prompt_tokens": stats["prompt_tokens"],
                            "completion_tokens": stats["completion_tokens"],
                            "latency": stats["latency"],
                            **additional,
                        })
                    except RateLimitError as e:
                        print(f"Error : {e}")
                        print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                        continue
                    except Exception as e:
                        print(f"ERROR at sample {idx}: {e}")
                        continue

                df_out = pd.DataFrame(rows)
                df_out.to_csv(output_file, index=False)
                print(f"✅ Saved {len(df_out)} rows to {output_file}")

                if case_name == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                else:
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

                print("\n=== Classification Report ===")
                print(classification_report(y_true, y_pred))

                print("\n=== Confusion Matrix ===")
                labels = sorted(set(y_true) | set(y_pred))
                conf_matrix = confusion_matrix(y_true, y_pred)
                print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

                accuracy = accuracy_score(y_true, y_pred)
                print(f"\n=== Accuracy for {case_name} | {person_key} | {setting} | {role_playing}: {accuracy:.3f}")



manipulation | profile26 | few_shot | passive: 100%|██████████| 500/500 [06:57<00:00,  1.20it/s]


✅ Saved 500 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile26_passive/results_manipulation_few_shot_prompt_short_3examples_binary.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.67      0.65      0.66       250
           1       0.66      0.68      0.67       250

    accuracy                           0.66       500
   macro avg       0.66      0.66      0.66       500
weighted avg       0.66      0.66      0.66       500


=== Confusion Matrix ===
     0    1
0  162   88
1   80  170

=== Accuracy for manipulation | profile26 | few_shot | passive: 0.664


manipulation | profile27 | few_shot | passive: 100%|██████████| 500/500 [06:02<00:00,  1.38it/s]


✅ Saved 500 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile27_passive/results_manipulation_few_shot_prompt_short_3examples_binary.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.67      0.65      0.66       250
           1       0.66      0.68      0.67       250

    accuracy                           0.67       500
   macro avg       0.67      0.67      0.67       500
weighted avg       0.67      0.67      0.67       500


=== Confusion Matrix ===
     0    1
0  163   87
1   79  171

=== Accuracy for manipulation | profile27 | few_shot | passive: 0.668


manipulation | profile28 | few_shot | passive: 100%|██████████| 500/500 [06:16<00:00,  1.33it/s]


✅ Saved 500 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile28_passive/results_manipulation_few_shot_prompt_short_3examples_binary.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.67      0.69      0.68       250
           1       0.68      0.66      0.67       250

    accuracy                           0.68       500
   macro avg       0.68      0.68      0.68       500
weighted avg       0.68      0.68      0.68       500


=== Confusion Matrix ===
     0    1
0  172   78
1   84  166

=== Accuracy for manipulation | profile28 | few_shot | passive: 0.676


manipulation | profile29 | few_shot | passive: 100%|██████████| 500/500 [05:54<00:00,  1.41it/s]

✅ Saved 500 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile29_passive/results_manipulation_few_shot_prompt_short_3examples_binary.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.68      0.63      0.66       250
           1       0.66      0.71      0.68       250

    accuracy                           0.67       500
   macro avg       0.67      0.67      0.67       500
weighted avg       0.67      0.67      0.67       500


=== Confusion Matrix ===
     0    1
0  158   92
1   73  177

=== Accuracy for manipulation | profile29 | few_shot | passive: 0.670
